In [24]:
import pandas as pd
import numpy as np
import seaborn as sns
import re

In [2]:
df_deepdive = pd.read_csv("/home/binperdok/KLTN2026/Data/data_label_deepdive/final_09_03_2026.csv")
df_market = pd.read_csv("/home/binperdok/KLTN2026/Data/data_label_market/final_marrket.csv")
df_usecase = pd.read_csv("/home/binperdok/KLTN2026/Data/data_label_usecase/final_usecase.csv")

In [3]:
df_deepdive.head(5)

,title,content,label
0,PRX Part 3 — Đào tạo một Text-to-Image Model t...,Chào mừng trở lại 👋 Trong hai bài viết trước (...,DEEP DIVE
1,Mixture of Experts (MoEs) trong Transformers,"Trong vài năm qua, việc mở rộng quy mô các mô ...",DEEP DIVE
2,Huấn luyện các mô hình AI với Unsloth và Huggi...,Bạn sẽ cầnChạy JobCài đặt SkillClaude CodeCode...,DEEP DIVE
3,Transformers.js v4 Preview: Hiện đã có mặt trê...,Cải thiện Hiệu suất & Thời gian chạy\nTái cấu ...,DEEP DIVE
4,Mô hình Holo2 mới của Công ty H dẫn đầu trong ...,Hai tháng kể từ khi phát hành đợt đầu tiên của...,DEEP DIVE


In [4]:
df_market.head()

,title,content,label
0,CEO Anthropic lên tiếng sau mâu thuẫn với Lầu ...,"""Chúng tôi thực hiện quyền của mình theo Tu ch...",MARKET SIGNALS
1,Xe vá ổ gà trong hai phút,"Sử dụng công nghệ DuraPatcher, xe Cimline P5 d...",NOISE
2,Chatbot Anthropic đứng đầu bảng ứng dụng miễn ...,"TheoBusiness Insider, tính đến cuối giờ chiều ...",MARKET SIGNALS
3,OpenAI tìm kiếm người không chuyên nhưng 'có gu',"Theo Altman, một trong những con đường tốt nhấ...",MARKET SIGNALS
4,ChatGPT chạm mốc 900 triệu người dùng,Thông báo mới cho thấy ChatGPT đang tiến gần h...,MARKET SIGNALS


In [5]:
df_usecase.head()

,title,content,label
0,Ứng Dụng AI Trong Y Tế: Cách Mạng Hóa Chăm Sóc...,Bạn có tin rằng AI hiện nay AI đã chẩn đoán un...,SOLUTIONS & USE CASES
1,Xu Hướng Ứng Dụng AI Trong Du Lịch: 20+ Cách T...,Ngành du lịch đang đối mặt với một thách thức ...,SOLUTIONS & USE CASES
2,Ứng dụng AI cho từng phòng ban: Giải pháp tối ...,"Theo tạp chí Forbes, có tới 83% công ty tuyên ...",SOLUTIONS & USE CASES
3,5 Ứng dụng AI trong kinh doanh giúp doanh nghi...,Trí tuệ nhân tạo (AI) đã không còn là công ngh...,SOLUTIONS & USE CASES
4,"Ứng dụng AI trong Bất động sản 2025: Xu hướng,...",Shark Hưng từng nhận định: “Mô hình môi giới B...,SOLUTIONS & USE CASES


In [6]:
data = pd.concat([df_usecase, df_deepdive , df_market], axis = 0, ignore_index=True)
data.head()

,title,content,label
0,Ứng Dụng AI Trong Y Tế: Cách Mạng Hóa Chăm Sóc...,Bạn có tin rằng AI hiện nay AI đã chẩn đoán un...,SOLUTIONS & USE CASES
1,Xu Hướng Ứng Dụng AI Trong Du Lịch: 20+ Cách T...,Ngành du lịch đang đối mặt với một thách thức ...,SOLUTIONS & USE CASES
2,Ứng dụng AI cho từng phòng ban: Giải pháp tối ...,"Theo tạp chí Forbes, có tới 83% công ty tuyên ...",SOLUTIONS & USE CASES
3,5 Ứng dụng AI trong kinh doanh giúp doanh nghi...,Trí tuệ nhân tạo (AI) đã không còn là công ngh...,SOLUTIONS & USE CASES
4,"Ứng dụng AI trong Bất động sản 2025: Xu hướng,...",Shark Hưng từng nhận định: “Mô hình môi giới B...,SOLUTIONS & USE CASES


In [7]:
data.label.value_counts()

label
MARKET SIGNALS           593
NOISE                    275
DEEP DIVE                257
SOLUTIONS & USE CASES    231
Name: count, dtype: int64

In [10]:
data.isnull().sum()

title       0
content     0
label      42
dtype: int64

In [12]:
data = data.dropna(subset = ['label'])
data.isnull().sum()

title      0
content    0
label      0
dtype: int64

In [13]:
(data == "").any().any()   

np.False_

In [16]:
data.head()

,title,content,label
0,08 cách tối ưu Chatbot hỗ trợ mang lại hiệu qu...,Chatbot đang trở nên phổ biến và trở thành một...,SOLUTIONS & USE CASES
1,Ngôi sao chip bán dẫn rời Mỹ về Đại học Thanh ...,"Su Fei hiện là ""Giáo sư danh dự"" tại trường Vi...",MARKET SIGNALS
2,Hàn Quốc ban hành luật AI toàn diện đầu tiên t...,"Ở Hàn Quốc, Luật Cơ bản về Phát triển AI và Th...",MARKET SIGNALS
3,Cuộc đua phổ cập AI trong doanh nghiệp Việt - ...,Doanh nghiệp Ấn Độ đào tạo AI quy mô lớn Đào t...,MARKET SIGNALS
4,Google công bố Nano Banana 2 cạnh tranh với Ch...,Nano Banana 2 không chỉ mang đến khả năng sáng...,MARKET SIGNALS


In [15]:
# Trộn toàn bộ dữ liệu, giữ nguyên số dòng
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

In [18]:
target_n = 275   # hoặc 250, 300... tùy bạn

# Tách ra 2 phần: MARKET SIGNALS và phần còn lại
market = data[data['label'] == 'MARKET SIGNALS']
others = data[data['label'] != 'MARKET SIGNALS']

# Lấy ngẫu nhiên target_n dòng từ MARKET SIGNALS
market_down = market.sample(n=target_n, random_state=42)

# Gộp lại
data_balanced = pd.concat([market_down, others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

data_balanced.label.value_counts()

label
MARKET SIGNALS           275
NOISE                    275
DEEP DIVE                257
SOLUTIONS & USE CASES    231
Name: count, dtype: int64

In [21]:
data_balanced.to_csv("/home/binperdok/KLTN2026/Data/FINAL_DATA.csv")

In [22]:
data['title'] = data['title'].fillna('')
data['content'] = data['content'].fillna('')

data['text'] = data['title'] + ' ' + data['content']

In [23]:
# xử lý đưa về chữ thường
data['text'] = data['text'].str.lower()

In [25]:
# xoa url, email, html tag
def remove_urls_emails_html(x):
    x = re.sub(r'http\S+|www\.\S+', ' ', x)        # URL
    x = re.sub(r'\S+@\S+\.\S+', ' ', x)           # email
    x = re.sub(r'<.*?>', ' ', x)                  # HTML
    return x

data['text'] = data['text'].apply(remove_urls_emails_html)

In [26]:
# Xoa ki tu dac biet
def remove_special_chars(x):
    x = re.sub(r'[^a-zA-ZÀ-ỹà-ỹ\s]', ' ', x)  # nếu có tiếng Việt
    return x

data['text'] = data['text'].apply(remove_special_chars)

In [27]:
#Xoa khong thua o dau va cuoi 
data['text'] = data['text'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [28]:
from underthesea import word_tokenize

def tokenize_text(s):
    s = str(s)
    return word_tokenize(s, format="text")

data['text_tok'] = data['text'].apply(tokenize_text)

In [ ]:
data.head() 

,title,content,label,text,text_tok
0,08 cách tối ưu Chatbot hỗ trợ mang lại hiệu qu...,Chatbot đang trở nên phổ biến và trở thành một...,SOLUTIONS & USE CASES,cách tối ưu chatbot hỗ trợ mang lại hiệu quả k...,cách tối_ưu_chatbot hỗ_trợ mang lại hiệu_quả k...
1,Ngôi sao chip bán dẫn rời Mỹ về Đại học Thanh ...,"Su Fei hiện là ""Giáo sư danh dự"" tại trường Vi...",MARKET SIGNALS,ngôi sao chip bán dẫn rời mỹ về đại học thanh ...,ngôi_sao chip bán_dẫn rời mỹ về đại_học thanh_...
2,Hàn Quốc ban hành luật AI toàn diện đầu tiên t...,"Ở Hàn Quốc, Luật Cơ bản về Phát triển AI và Th...",MARKET SIGNALS,hàn quốc ban hành luật ai toàn diện đầu tiên t...,hàn_quốc ban_hành luật ai toàn_diện đầu_tiên t...
3,Cuộc đua phổ cập AI trong doanh nghiệp Việt - ...,Doanh nghiệp Ấn Độ đào tạo AI quy mô lớn Đào t...,MARKET SIGNALS,cuộc đua phổ cập ai trong doanh nghiệp việt ca...,cuộc đua phổ_cập ai trong doanh_nghiệp việt_ca...
4,Google công bố Nano Banana 2 cạnh tranh với Ch...,Nano Banana 2 không chỉ mang đến khả năng sáng...,MARKET SIGNALS,google công bố nano banana cạnh tranh với chat...,google công_bố nano banana cạnh_tranh với chat...


In [30]:
import unicodedata2

def normalize_unicode(text):
    return unicodedata2.normalize('NFC', str(text))

data['text_tok'] = data['text_tok'].apply(normalize_unicode)

In [31]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['label_enc'] = le.fit_transform(data['label'])

print(dict(zip(le.classes_, le.transform(le.classes_))))
# Ví dụ: {'DEEP DIVE': 0, 'MARKET SIGNALS': 1, 'NOISE': 2, 'SOLUTIONS & USE CASES': 3}

{'DEEP DIVE': np.int64(0), 'MARKET SIGNALS': np.int64(1), 'NOISE': np.int64(2), 'SOLUTIONS & USE CASES': np.int64(3)}


In [32]:
data[['text_tok', 'label', 'label_enc']].to_csv(
    "/home/binperdok/KLTN2026/Data/PROCESSED_DATA.csv", index=False
)